# 06 — Benchmark Comparison (Phase 1 Results)

This notebook loads the actual Phase 1 experimental results from `logs/thesis/headline_0200-27042026/` and reproduces the key figures from the thesis:

1. **AgentDojo PSR** (Poisoning Success Rate) — no_defense vs all baselines vs SPQ, across 3 frontier models.
2. **Bootstrap CI visualization** — 95% confidence intervals per defense per model.
3. **Tier-3 ASR table** — all-zeros result for frontier models on static content safety benchmarks.

## Experiment configuration (Phase 1)

| Parameter | Value |
|---|---|
| Victim models | GPT-4o-2024-11-20, Claude-Sonnet-4, Llama-3.3-70B |
| Seeds | 3 (0, 1, 2) |
| Goals per seed | 8 |
| Tiers | 1 (agentic) + 3 (content safety) |
| Judge | Gemini-2.5-Flash via OpenRouter |

**Thesis context** (§6 *Results*): Phase 1 establishes the headline numbers. The primary finding is that SPQ achieves competitive PSR reduction on AgentDojo with a very small additional overhead versus prompt-level baselines, while providing the only **provable** defense against adaptive tool-call attacks.

In [ ]:
# ── CONFIG CELL ───────────────────────────────────────────────────────
LOG_DIR = '/Users/tlysu/ucu/diploma/logs/thesis/headline_0200-27042026'
BENCHMARK = 'tier1/agentdojo.json'    # primary AgentDojo benchmark
PSR_METRIC = 'psr_judge'              # canonical judged PSR
# ─────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os, json
sys.path.insert(0, '/Users/tlysu/ucu/diploma')

import matplotlib
matplotlib.use('Agg')  # change to 'inline' for interactive Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("Imports OK.")

## Load Phase 1 aggregate results

In [ ]:
MODEL_DIRS = {
    'GPT-4o':       'model_openai_gpt-4o-2024-11-20',
    'Claude-Sonnet':'model_anthropic_claude-sonnet-4',
    'Llama-3.3-70B':'model_meta-llama_llama-3.3-70b-instruct',
}

# Load PSR data
psr_data = {}
for model_name, model_dir in MODEL_DIRS.items():
    path = os.path.join(LOG_DIR, model_dir, 'aggregate', 'psr_with_ci.json')
    with open(path) as f:
        d = json.load(f)
    psr_data[model_name] = d['psr_with_ci'].get(BENCHMARK, {})
    print(f"Loaded {model_name}: {len(psr_data[model_name])} defense entries")

# Load ASR data (Tier-3)
asr_data = {}
for model_name, model_dir in MODEL_DIRS.items():
    path = os.path.join(LOG_DIR, model_dir, 'aggregate', 'asr_with_ci.json')
    with open(path) as f:
        d = json.load(f)
    asr_data[model_name] = d['asr_with_ci']

print("\nASR benchmarks (Tier-3):", list(list(asr_data.values())[0].keys()))

## AgentDojo PSR — defense comparison table

In [ ]:
# Collect all defenses from the first model
defenses_ordered = list(list(psr_data.values())[0].keys())

# Defense display names
DEFENSE_LABELS = {
    'no_defense':           'No Defense',
    'StruQDefense':         'StruQ',
    'DataSentinelDefense':  'DataSentinel',
    'MELONDefense':         'MELON',
    'AMemGuard':            'AMemGuard',
    'SmoothLLMDefense':     'SmoothLLM',
    'PromptGuard2Defense':  'PromptGuard2',
    'PACEDefense':           'SPQ (ours)',
}

print(f"AgentDojo PSR (metric: {PSR_METRIC})")
print(f"{'Defense':<18} ", end="")
for m in MODEL_DIRS:
    print(f"{m:>20}", end="")
print()
print("-" * (18 + 20 * len(MODEL_DIRS) + 2))

table_rows = []
for defense in defenses_ordered:
    label = DEFENSE_LABELS.get(defense, defense)
    row = [label]
    print(f"{label:<18} ", end="")
    for model_name in MODEL_DIRS:
        entry = psr_data[model_name].get(defense, {})
        metric = entry.get(PSR_METRIC, entry.get('psr', {}))
        pt  = metric.get('point', float('nan'))
        lo  = metric.get('ci95_low', pt)
        hi  = metric.get('ci95_high', pt)
        cell = f"{pt:.3f} [{lo:.3f},{hi:.3f}]"
        print(f"{cell:>20}", end="")
        row.append((pt, lo, hi))
    print()
    table_rows.append(row)

## Bar chart with bootstrap CIs

In [ ]:
os.makedirs('/Users/tlysu/ucu/diploma/notebooks/figures', exist_ok=True)

models_list  = list(MODEL_DIRS.keys())
n_defenses   = len(defenses_ordered)
n_models     = len(models_list)

x      = np.arange(n_defenses)
width  = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(12, 5))

for mi, (model_name, color) in enumerate(zip(models_list, colors)):
    pts, lows, highs = [], [], []
    for defense in defenses_ordered:
        entry  = psr_data[model_name].get(defense, {})
        metric = entry.get(PSR_METRIC, entry.get('psr', {}))
        pt  = metric.get('point', 0.0)
        lo  = metric.get('ci95_low', pt)
        hi  = metric.get('ci95_high', pt)
        pts.append(pt)
        lows.append(pt - lo)
        highs.append(hi - pt)

    bars = ax.bar(
        x + mi * width, pts, width,
        label=model_name, color=color, alpha=0.85, zorder=3
    )
    ax.errorbar(
        x + mi * width, pts,
        yerr=[lows, highs],
        fmt='none', ecolor='black', elinewidth=1.5, capsize=4, zorder=4
    )

# Highlight SPQ bar group
spq_idx = defenses_ordered.index('PACEDefense') if 'PACEDefense' in defenses_ordered else None
if spq_idx is not None:
    ax.axvspan(spq_idx - 0.35, spq_idx + 0.85, alpha=0.08, color='green', zorder=1)
    ax.text(spq_idx + 0.2, ax.get_ylim()[1] * 0.98 if ax.get_ylim()[1] > 0 else 0.15,
            'SPQ', ha='center', fontsize=9, color='darkgreen', style='italic')

defense_labels = [DEFENSE_LABELS.get(d, d) for d in defenses_ordered]
ax.set_xticks(x + width)
ax.set_xticklabels(defense_labels, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Poisoning Success Rate (PSR)', fontsize=10)
ax.set_title('AgentDojo PSR — Phase 1 Headline Results (3 seeds, 95% bootstrap CI)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(0, min(0.3, max(0.01, ax.get_ylim()[1]) * 1.25))

plt.tight_layout()
fig_path = '/Users/tlysu/ucu/diploma/notebooks/figures/06_agentdojo_psr.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Figure saved to {fig_path}")
plt.show()

## Tier-3 ASR table — all-zeros result

Tier-3 benchmarks (AdvBench, HarmBench, JailbreakBench) test whether frontier models comply with explicit harmful requests. The result is ASR=0 for all models and defenses — frontier RLHF is sufficient for this class.

**Why this matters for the thesis**: it confirms that the hard problem is *agentic* attacks (Tier-1), not *content* attacks (Tier-3). Defense research should focus on the agentic threat model.

In [ ]:
print("Tier-3 ASR (no_defense, StaticAttackSuite) — judged ASR")
print()
print(f"{'Benchmark':<35} ", end="")
for m in MODEL_DIRS:
    print(f"{m:>20}", end="")
print()
print("-" * (35 + 20 * len(MODEL_DIRS) + 2))

tier3_benchmarks = [
    ('tier3/advbench.json',       'AdvBench'),
    ('tier3/harmbench.json',      'HarmBench'),
    ('tier3/jailbreakbench.json', 'JailbreakBench'),
]

for bench_key, bench_name in tier3_benchmarks:
    print(f"{bench_name:<35} ", end="")
    for model_name in MODEL_DIRS:
        bench_data = asr_data[model_name].get(bench_key, {})
        nd = bench_data.get('no_defense', {})
        static = nd.get('StaticAttackSuite', nd.get('HumanRedTeamAttack', {}))
        pt = static.get('point', 0.0)
        print(f"{pt:>20.3f}", end="")
    print()

In [ ]:
# Show all attack families for AdvBench (all should be 0.0)
print("AdvBench ASR — all defenses and attack families for GPT-4o:")
print()
advbench_gpt4o = asr_data['GPT-4o'].get('tier3/advbench.json', {})

for defense_key, defense_results in advbench_gpt4o.items():
    print(f"  Defense: {defense_key}")
    for attack_name, attack_data in defense_results.items():
        pt = attack_data.get('point', '?')
        print(f"    {attack_name:<35} ASR={pt:.3f}")
    print()

## Bootstrap CI visualization — SPQ focus

In [ ]:
# CI plot: for each model, show all defenses with error bars (PSR_METRIC)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (model_name, color) in zip(axes, zip(models_list, colors)):
    pts, lows, highs, labels = [], [], [], []
    for defense in defenses_ordered:
        entry  = psr_data[model_name].get(defense, {})
        metric = entry.get(PSR_METRIC, entry.get('psr', {}))
        pt = metric.get('point', 0.0)
        lo = metric.get('ci95_low', pt)
        hi = metric.get('ci95_high', pt)
        pts.append(pt)
        lows.append(pt - lo)
        highs.append(hi - pt)
        labels.append(DEFENSE_LABELS.get(defense, defense))

    y_pos = range(len(defenses_ordered))
    bar_colors = ['#2ca02c' if d == 'PACEDefense' else color for d in defenses_ordered]
    ax.barh(y_pos, pts, xerr=[lows, highs],
            color=bar_colors, alpha=0.8, ecolor='black', capsize=4)
    ax.axvline(pts[0], linestyle='--', color='gray', linewidth=1, alpha=0.7)  # no_defense baseline
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('PSR (judged)', fontsize=9)
    ax.set_title(model_name, fontsize=9)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('AgentDojo PSR per defense — 95% bootstrap CI (dashed = no-defense baseline)', fontsize=10)
plt.tight_layout()
fig_path = '/Users/tlysu/ucu/diploma/notebooks/figures/06_psr_ci_forest.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Forest plot saved to {fig_path}")
plt.show()

## Seed-level variance

In [ ]:
# Show per-seed PSR points for SPQ vs no_defense on AgentDojo
print("Per-seed PSR (SPQ vs no_defense) — AgentDojo:")
print()
print(f"{'Model':<22} {'Defense':<16} {'Seed 0':>8} {'Seed 1':>8} {'Seed 2':>8} {'Mean':>8}")
print("-" * 70)

for model_name in models_list:
    for defense in ['no_defense', 'PACEDefense']:
        entry  = psr_data[model_name].get(defense, {})
        metric = entry.get(PSR_METRIC, entry.get('psr', {}))
        seeds  = metric.get('seed_points', [])
        mean   = metric.get('point', float('nan'))
        seed_str = "".join(f"{s:>8.3f}" for s in seeds)
        label  = DEFENSE_LABELS.get(defense, defense)
        print(f"{model_name:<22} {label:<16} {seed_str} {mean:>8.3f}")

## Summary statistics

In [ ]:
print("Phase 1 Headline Summary")
print("=" * 55)
print()

for model_name in models_list:
    nd_entry = psr_data[model_name].get('no_defense', {})
    nd_psr   = nd_entry.get(PSR_METRIC, nd_entry.get('psr', {})).get('point', float('nan'))

    spq_entry = psr_data[model_name].get('PACEDefense', {})
    spq_psr   = spq_entry.get(PSR_METRIC, spq_entry.get('psr', {})).get('point', float('nan'))
    spq_lo    = spq_entry.get(PSR_METRIC, spq_entry.get('psr', {})).get('ci95_low', spq_psr)
    spq_hi    = spq_entry.get(PSR_METRIC, spq_entry.get('psr', {})).get('ci95_high', spq_psr)

    delta     = nd_psr - spq_psr
    delta_pct = (delta / nd_psr * 100) if nd_psr > 0 else 0.0

    # Best baseline
    best_base_psr  = float('inf')
    best_base_name = ''
    for d in defenses_ordered:
        if d in ('no_defense', 'PACEDefense'):
            continue
        entry = psr_data[model_name].get(d, {})
        pt = entry.get(PSR_METRIC, entry.get('psr', {})).get('point', float('inf'))
        if pt < best_base_psr:
            best_base_psr  = pt
            best_base_name = DEFENSE_LABELS.get(d, d)

    print(f"Model: {model_name}")
    print(f"  No defense PSR         : {nd_psr:.4f}")
    print(f"  Best baseline PSR      : {best_base_psr:.4f}  ({best_base_name})")
    print(f"  SPQ PSR                : {spq_psr:.4f}  95% CI [{spq_lo:.4f}, {spq_hi:.4f}]")
    print(f"  SPQ vs no_defense      : Δ={delta:+.4f}  ({delta_pct:+.1f}%)")
    print()

## Key takeaways

1. **Tier-3 (content safety)**: all defenses achieve ASR=0 on frontier models. Existing RLHF is sufficient. This confirms the thesis's focus on Tier-1 agentic attacks.

2. **AgentDojo (agentic)**: no-defense PSR is 9–12% across models. SPQ achieves the lowest or near-lowest PSR across all three models, with competitive 95% CIs.

3. **Confidence intervals are wide** due to small per-seed sample sizes (8 goals). Phase 2 (cheap models, 5 seeds, 12 goals) provides tighter estimates.

4. **SPQ's advantage is architectural, not metric-marginal**: unlike baselines that filter text, SPQ provides a *provable* CFI guarantee — even if the PSR numbers overlap within CIs, SPQ is the only defense that prevents tool calls not in the shadow plan by construction.